In [ ]:
import os
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS
import pandas as pd

In [ ]:
# --- CONFIGURACIÓN DE INFLUXDB (v2) ---
env_file = find_dotenv(".env", usecwd=True)
if not env_file:
  for candidate in (
      Path.cwd() / ".env",
      Path.cwd().parent / ".env",
      Path.cwd().parent.parent / ".env",
  ):
    if candidate.is_file():
      env_file = str(candidate)
      break

if not env_file:
  raise FileNotFoundError("No se encontró el archivo .env")

load_dotenv(env_file, override=True)


def required_env(name):
  value = os.getenv(name)
  if not value:
    raise RuntimeError(f"Falta la variable {name} en {env_file}")
  return value


INFLUX_URL = required_env("INFLUX_URL")
INFLUX_TOKEN = required_env("INFLUX_TOKEN")
INFLUX_ORG = required_env("INFLUX_ORG")
INFLUX_BUCKET_DOBOT = required_env("INFLUX_BUCKET_DOBOT")

# --- PARÁMETROS DEL DATASET ---
CSV_FILE = "../datos/dataset_dobot_cr5a.csv"
DEVICE_NAME = "Dobot_CR5A"  # Se asignará como Tag

# --- MAPEO DE ESTADOS DEL ROBOT DOBOT CR SERIES ---
ROBOT_MODE_MAP = {
    1: "Inicializando",
    2: "Frenos Abiertos",
    3: "Deshabilitado",
    4: "Habilitado / En espera",
    5: "Guiado Manual (Drag)",
    6: "En Ejecución",
    7: "Grabando",
    8: "Pausado",
    9: "Modo Jog",
    10: "Error / Colisión",
    11: "Paso a Paso",
}

In [ ]:
print(f"Cargando dataset desde '{CSV_FILE}'...")

try:
  df = pd.read_csv(CSV_FILE, engine="python", on_bad_lines="skip")

  # Parsear la estampa de tiempo a objetos datetime nativos
  df["timestamp"] = pd.to_datetime(df["timestamp"])

  # Si el CSV no incluye 'robot_mode_desc', la calculamos automáticamente
  if "robot_mode_desc" not in df.columns:
    df["robot_mode_desc"] = df["robot_mode"].apply(
        lambda code: ROBOT_MODE_MAP.get(
            int(code), f"Desconocido ({code})" if pd.notnull(code) else "N/A"
        )
    )

  print(f"Registros cargados correctamente: {len(df)} filas.")
  print("\nVista previa de los datos (con estado traducido):")
  display(df.tail(3))

except FileNotFoundError:
  print(f"Error: No se encontró el archivo '{CSV_FILE}'. Revisa la ruta.")

Cargando dataset desde '../datos/dataset_dobot_cr5a.csv'...
Registros cargados: 3740 filas.

Vista previa de los datos:


,timestamp,robot_mode,pos_x,pos_y,pos_z,rot_rx,rot_ry,rot_rz,j1_temp,j2_temp,j3_temp,j4_temp,j5_temp,j6_temp,j1_current,j2_current,j3_current,j4_current,j5_current,j6_current
3737,2026-10-13 08:22:41.940300,4,0.07,-90.1,-0.11,90.04,91.34,-0.0,-470.4,288.2,179.5,-0.1,88.1,0.0,0.276,-0.266,0.056,-0.000,-0.068,0.0
3738,2026-10-13 08:22:42.444783,4,0.07,-90.1,-0.11,90.04,91.34,-0.0,-470.4,288.2,179.5,-0.1,88.1,0.0,0.204,-0.111,0.041,-0.000,0.000,0.0
3739,2026-10-13 08:22:42.950905,4,0.07,-90.1,-0.11,90.04,91.34,-0.0,-470.4,288.2,179.5,-0.1,88.1,0.3,-0.133,-0.499,0.055,-0.068,-0.000,0.0


In [ ]:
# Inicializar cliente de InfluxDB y la API de escritura síncrona
client = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
write_api = client.write_api(write_options=SYNCHRONOUS)

puntos_batch = []
tamanio_lote = 500  # Envío agrupado de 500 filas por llamada para maximizar velocidad
total_filas = len(df)

print(f"Subiendo {total_filas} registros al bucket '{INFLUX_BUCKET_DOBOT}'...")

for index, row in df.iterrows():
  mode_code = int(row["robot_mode"])
  mode_desc = str(row["robot_mode_desc"])

  punto = (
      Point("dobot_telemetry")  # Measurement
      # --- TAGS (Metadatos indexados para filtrado rápido en Grafana) ---
      .tag("device_name", DEVICE_NAME)
      .tag("model", "CR5A")
      .tag("robot_mode_desc", mode_desc)
      # --- FIELDS: Posición Cartesiana de la Herramienta (mm / deg) ---
      .field("pos_x", float(row["pos_x"]))
      .field("pos_y", float(row["pos_y"]))
      .field("pos_z", float(row["pos_z"]))
      .field("rot_rx", float(row["rot_rx"]))
      .field("rot_ry", float(row["rot_ry"]))
      .field("rot_rz", float(row["rot_rz"]))
      # --- FIELDS: Temperaturas de los 6 Servomotores (°C) ---
      .field("j1_temp", float(row["j1_temp"]))
      .field("j2_temp", float(row["j2_temp"]))
      .field("j3_temp", float(row["j3_temp"]))
      .field("j4_temp", float(row["j4_temp"]))
      .field("j5_temp", float(row["j5_temp"]))
      .field("j6_temp", float(row["j6_temp"]))
      # --- FIELDS: Corrientes de Consumo de los 6 Ejes (A) ---
      .field("j1_current", float(row["j1_current"]))
      .field("j2_current", float(row["j2_current"]))
      .field("j3_current", float(row["j3_current"]))
      .field("j4_current", float(row["j4_current"]))
      .field("j5_current", float(row["j5_current"]))
      .field("j6_current", float(row["j6_current"]))
      # --- FIELDS: Estado Operativo del Robot ---
      .field("robot_mode", mode_code)
      .field("robot_mode_text", mode_desc)
      # --- TIMESTAMP ---
      .time(row["timestamp"], WritePrecision.NS)
  )
  puntos_batch.append(punto)

  # Enviar al servidor al completar un lote
  if len(puntos_batch) >= tamanio_lote:
    write_api.write(
        bucket=INFLUX_BUCKET_DOBOT, org=INFLUX_ORG, record=puntos_batch
    )
    puntos_batch = []
    print(f"Registros cargados: {index + 1}/{total_filas}...")

# Enviar los datos sobrantes del último lote
if puntos_batch:
  write_api.write(
      bucket=INFLUX_BUCKET_DOBOT, org=INFLUX_ORG, record=puntos_batch
  )

print(f"\n¡Migración completada! {total_filas} filas insertadas en InfluxDB.")
client.close()

Subiendo 987 registros al bucket 'dobot'...
Registros cargados: 500/987...

¡Migración completada! 987 filas insertadas en InfluxDB.
